# Sentinel-1 Acquisition for RoughNet

This notebook queries the Copernicus Data Space Ecosystem (CDSE) for Sentinel-1 GRD products covering the existing LiDAR study patches, selects scenes close to each LiDAR acquisition date, downloads the SAFE archives, and extracts them.

Credentials are read from environment variables. Do not place a password or access token in this notebook. This notebook is intentionally unexecuted; run it on the GPU machine only when the data paths and credentials are configured.

## Configuration

The AOI is built from existing `lidar_patch_*.tif` files. The resulting products are saved under `raw_data/<region>_sentinel1_downloads/`.

In [ ]:
import os
import time
import json
import zipfile
import datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.warp import transform_geom
from shapely.geometry import box, shape
from shapely.ops import unary_union

In [ ]:
REPO_DIR = Path('/Users/jessica/Desktop/project/Michel/RoughNet')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'tuk'  # change to pondinlet or cambridge
LIDAR_DIR = INPUT_DIR / f'lidar_patches_{REGION}'
RAW_DIR = REPO_DIR / 'raw_data' / f'{REGION}_sentinel1_downloads'
DATE_BY_REGION = {
    'pondinlet': dt.date(2024, 4, 26),
    'cambridge': dt.date(2024, 4, 18),
    'tuk': dt.date(2024, 4, 16),
}
SEARCH_DAYS = 14
MAX_PRODUCTS = 6
PRODUCT_TYPES = ['EW_GRDM_1S', 'IW_GRDH_1S']
CDSE_USERNAME = os.environ['CDSE_USERNAME']
CDSE_PASSWORD = os.environ['CDSE_PASSWORD']
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Build the search AOI

Sampling the patch bounds keeps the CDSE query geometry compact while still covering the LiDAR study area.

In [ ]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
AOI_WKT = aoi.wkt
print(f'AOI vertices: {len(aoi.exterior.coords)}')

## Authenticate and query CDSE

In [ ]:
TOKEN_URL = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
CATALOGUE_URL = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'

def get_access_token(username, password):
    response = requests.post(TOKEN_URL, data={
        'grant_type': 'password',
        'username': username,
        'password': password,
        'client_id': 'cdse-public',
    }, timeout=60)
    response.raise_for_status()
    return response.json()['access_token']

access_token = get_access_token(CDSE_USERNAME, CDSE_PASSWORD)
headers = {'Authorization': f'Bearer {access_token}'}
start = DATE_BY_REGION[REGION] - dt.timedelta(days=SEARCH_DAYS)
end = DATE_BY_REGION[REGION] + dt.timedelta(days=SEARCH_DAYS)
product_filter = ' or '.join([f
 + 
 + '
 for kind in PRODUCT_TYPES])
odata_filter = (
    "Collection/Name eq 'SENTINEL-1' and "
    f"ContentDate/Start ge {start.isoformat()}T00:00:00.000Z and "
    f"ContentDate/Start le {end.isoformat()}T23:59:59.999Z and "
    f"({product_filter})"
)
params = {'$filter': odata_filter, '$top': 200, '$orderby': 'ContentDate/Start asc'}
response = requests.get(CATALOGUE_URL, params=params, headers=headers, timeout=120)
response.raise_for_status()
products = pd.DataFrame(response.json().get('value', []))
print(f'Catalogue products returned: {len(products)}')

## Select, download, and extract products

Selection is based on acquisition date proximity. Inspect the resulting table before downloading at scale.

In [ ]:
if products.empty:
    raise RuntimeError('No Sentinel-1 products matched the query.')
products['acquisition_date'] = pd.to_datetime(products['ContentDate'].map(lambda x: x['Start'])).dt.date
products['date_distance_days'] = products['acquisition_date'].map(lambda x: abs((x - DATE_BY_REGION[REGION]).days))
selected = products.sort_values(['date_distance_days', 'acquisition_date']).head(MAX_PRODUCTS).copy()
display(selected[['Id', 'Name', 'acquisition_date', 'date_distance_days']])

def download_product(row, output_dir, token):
    product_id = row['Id']
    name = row['Name']
    zip_path = output_dir / f'{name}.zip'
    if zip_path.exists() and zip_path.stat().st_size > 0:
        return zip_path
    url = f'https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value'
    with requests.get(url, headers={'Authorization': f'Bearer {token}'}, stream=True, timeout=300) as r:
        r.raise_for_status()
        with zip_path.open('wb') as handle:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return zip_path

downloaded = []
for _, row in selected.iterrows():
    path = download_product(row, RAW_DIR, access_token)
    downloaded.append(path)
    print(path.name, path.stat().st_size, 'bytes')

for zip_path in downloaded:
    extract_dir = RAW_DIR / zip_path.stem
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)
    print('Extracted:', extract_dir)

## Output contract

The next notebook expects one or more extracted `*.SAFE` directories under `RAW_DIR`. It reads VV/VH measurement TIFFs and their calibration XML files from those SAFE products.